# Test Mistral OCR dengan CER dan WER (Versi Batch Bertahap)

Notebook ini menguji performa Mistral OCR dengan menghitung Character Error Rate (CER) dan Word Error Rate (WER) untuk semua gambar di folder 'tes gambar', dengan cara memproses satu per satu untuk menghindari rate limit.

In [1]:
import os
import sys
import json
import time
from pathlib import Path

# Tambahkan project root ke path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from services.ocr_service import OCRService

print("✅ Library berhasil diimport!")

✅ Library berhasil diimport!


## 1. Definisikan Ground Truth dan Fungsi Perhitungan CER/WER

In [2]:
ground_truth = """28 - 11 - 2029

Nama sales = Budi Santoso

Ambil barang
lolipop 100 ctn
meses 20 ctn
miksu 50 ctn

30 - 11 - 2029

Nama = Budi Santoso

Bayar cicilan : Rp 73.600.000

02 - 12 - 2029

Nama = Budi Santoso

Bayar cicilan : Rp 10.000.000"""

def calculate_cer(reference, hypothesis):
    """Calculate Character Error Rate (CER)."""
    ref = list(reference)
    hyp = list(hypothesis)
    
    # Create distance matrix
    d = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    
    for i in range(len(ref) + 1):
        d[i][0] = i
    for j in range(len(hyp) + 1):
        d[0][j] = j
    
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(
                d[i-1][j] + 1,        # deletion
                d[i][j-1] + 1,        # insertion
                d[i-1][j-1] + cost    # substitution
            )
    
    return d[len(ref)][len(hyp)] / len(ref) if len(ref) > 0 else 1.0


def calculate_wer(reference, hypothesis):
    """Calculate Word Error Rate (WER)."""
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    
    # Create distance matrix
    d = [[0] * (len(hyp_words) + 1) for _ in range(len(ref_words) + 1)]
    
    for i in range(len(ref_words) + 1):
        d[i][0] = i
    for j in range(len(hyp_words) + 1):
        d[0][j] = j
    
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            cost = 0 if ref_words[i-1] == hyp_words[j-1] else 1
            d[i][j] = min(
                d[i-1][j] + 1,        # deletion
                d[i][j-1] + 1,        # insertion
                d[i-1][j-1] + cost    # substitution
            )
    
    return d[len(ref_words)][len(hyp_words)] / len(ref_words) if len(ref_words) > 0 else 1.0

print("✅ Ground truth dan fungsi perhitungan siap!")

✅ Ground truth dan fungsi perhitungan siap!


## 2. Inisialisasi OCR Service

In [3]:
ocr_service = OCRService()
ocr_service.load_model()
print("✅ OCR Service siap!")

✅ OCR Service siap!


## 3. Fungsi untuk Memproses Satu Gambar

In [4]:
def process_single_image(img_number):
    """Memproses satu gambar dan menyimpan hasilnya ke file JSON."""
    img_filename = f"{img_number}.jpg"
    tes_gambar_dir = project_root / "tes gambar"
    img_path = str(tes_gambar_dir / img_filename)
    
    if not os.path.exists(img_path):
        print(f"❌ Error: Gambar {img_filename} tidak ditemukan di {tes_gambar_dir}")
        return False
    
    print(f"\nMemproses gambar {img_filename}...")
    print("=" * 80)
    
    # Ekstrak teks
    extracted_text = ocr_service.extract_text(img_path)
    
    # Hitung CER dan WER
    cer = calculate_cer(ground_truth, extracted_text)
    wer = calculate_wer(ground_truth, extracted_text)
    
    print(f"CER: {cer:.2%}")
    print(f"WER: {wer:.2%}")
    print(f"\nTeks diekstrak:\n{extracted_text}")
    print("-" * 80)
    
    # Simpan hasil ke file JSON
    results_dir = project_root / "ocr_results"
    results_dir.mkdir(exist_ok=True)
    result_file = results_dir / f"result_{img_number}.json"
    
    result_data = {
        "filename": img_filename,
        "cer": cer,
        "wer": wer,
        "extracted_text": extracted_text
    }
    
    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(result_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Hasil disimpan ke: {result_file}")
    return True

print("✅ Fungsi process_single_image siap!")

✅ Fungsi process_single_image siap!


## 4. Proses Gambar Satu Per Satu (Dengan Delay Opsional)

Jalankan sel berikut untuk memproses gambar satu per satu. Setiap selesai 4 gambar, akan ada delay 1 menit untuk menghindari rate limit!

In [5]:
def process_images_with_delay(start_num, end_num, delay_after=4, delay_seconds=60):
    """Memproses gambar dengan delay otomatis."""
    count = 0
    for img_num in range(start_num, end_num + 1):
        process_single_image(img_num)
        count += 1
        
        # Delay setelah N gambar
        if count % delay_after == 0 and img_num != end_num:
            print("\n" + "=" * 100)
            print(f"✅ Sudah memproses {count} gambar. Delay {delay_seconds} detik untuk menghindari rate limit...".center(100))
            print("=" * 100)
            time.sleep(delay_seconds)

# Contoh: Proses gambar 6-9 dengan delay 1 menit setiap 4 gambar
process_images_with_delay(6, 9, delay_after=4, delay_seconds=60)


Memproses gambar 6.jpg...
CER: 4.29%
WER: 14.29%

Teks diekstrak:
28 - 11 - 2029

Nama sales = Budi Santoso

Ambil barang
- lolipop 100 ctn
- meses 20 ctn
- miksu 50 ctn

30 - 11 - 2029

Nama = Budi Santoso

Bayar cicilan : Rp. 3.600.000

02 - 12 - 2029

Nama = Budi Santoso

Bayar Cicilan : Rp. 10.000.000
--------------------------------------------------------------------------------

✅ Hasil disimpan ke: C:\Users\FINN\Documents\BOT\ocr_results\result_6.json

Memproses gambar 7.jpg...
CER: 15.02%
WER: 22.45%

Teks diekstrak:
28 - 11 - 2029

Nama : Budi Santoso
Ambil barang :
- Wilio 100 ctn
- Beng-beng 20 ctn
- Messi 50 ctn

30 - 11 - 2029

Nama : Budi Santoso
Bayar cicilan : Rp 73.600.000

02 - 12 - 2029

Nama : Budi Santoso
Bayar cicilan : Rp 10.000.000
--------------------------------------------------------------------------------

✅ Hasil disimpan ke: C:\Users\FINN\Documents\BOT\ocr_results\result_7.json

Memproses gambar 8.jpg...
CER: 8.58%
WER: 20.41%

Teks diekstrak:
28 - 11 

## 5. Agregasi Semua Hasil dan Tarik Kesimpulan

In [6]:
def aggregate_results():
    """Menggabungkan semua hasil dan menampilkan ringkasan."""
    results_dir = project_root / "ocr_results"
    
    if not results_dir.exists():
        print(f"❌ Error: Folder {results_dir} tidak ditemukan! Jalankan terlebih dahulu process_single_image untuk setiap gambar.")
        return
    
    # Baca semua file result
    all_results = []
    for i in range(1, 10):
        result_file = results_dir / f"result_{i}.json"
        if result_file.exists():
            with open(result_file, "r", encoding="utf-8") as f:
                data = json.load(f)
                all_results.append(data)
                print(f"[OK] Memuat hasil untuk {data['filename']}")
        else:
            print(f"[WARN] Hasil untuk {i}.jpg belum ditemukan!")
    
    if not all_results:
        print("\n❌ Tidak ada hasil yang bisa diolah!")
        return
    
    # Hitung total dan rata-rata
    total_cer = sum(r["cer"] for r in all_results)
    total_wer = sum(r["wer"] for r in all_results)
    avg_cer = total_cer / len(all_results)
    avg_wer = total_wer / len(all_results)
    
    # Cari performa terbaik dan terburuk
    best_cer = min(all_results, key=lambda x: x["cer"])
    worst_cer = max(all_results, key=lambda x: x["cer"])
    best_wer = min(all_results, key=lambda x: x["wer"])
    worst_wer = max(all_results, key=lambda x: x["wer"])
    
    # Tampilkan ringkasan
    print("\n" + "=" * 100)
    print("RINGKASAN AKHIR PENGUJIAN OCR MISTRAL (9 GAMBAR)".center(100))
    print("=" * 100)
    print(f"\n1. JUMLAH GAMBAR YANG DIUJI: {len(all_results)} dari 9")
    print("\n2. DETAIL HASIL PER GAMBAR:")
    print("-" * 100)
    print(f"{'NAMA FILE':<15} {'CER':<12} {'WER':<12}")
    print("-" * 100)
    for r in all_results:
        print(f"{r['filename']:<15} {r['cer']:>8.2%}   {r['wer']:>8.2%}")
    print("-" * 100)
    print(f"\n3. RATA-RATA:")
    print(f"   - Rata-rata CER: {avg_cer:.2%}")
    print(f"   - Rata-rata WER: {avg_wer:.2%}")
    print("\n4. PERFORMA TERBAIK:")
    print(f"   - CER Terendah: {best_cer['filename']} ({best_cer['cer']:.2%})")
    print(f"   - WER Terendah: {best_wer['filename']} ({best_wer['wer']:.2%})")
    print("\n5. PERFORMA TERBURUK:")
    print(f"   - CER Tertinggi: {worst_cer['filename']} ({worst_cer['cer']:.2%})")
    print(f"   - WER Tertinggi: {worst_wer['filename']} ({worst_wer['wer']:.2%})")
    print("\n" + "=" * 100)
    print("KESIMPULAN".center(100))
    print("=" * 100)
    print(f"\nBerdasarkan pengujian {len(all_results)} gambar dari folder 'tes gambar':")
    if avg_cer < 0.30:
        print("[SANGAT BAIK] Performa OCR Mistral SANGAT BAIK dengan rata-rata CER di bawah 30%")
    elif avg_cer < 0.50:
        print("[CUKUP BAIK] Performa OCR Mistral CUKUP BAIK dengan rata-rata CER di bawah 50%")
    elif avg_cer < 0.70:
        print("[CUKUP] Performa OCR Mistral CUKUP dengan rata-rata CER di bawah 70%")
    else:
        print("[PERLU DIPERBAIKI] Performa OCR Mistral PERLU DIPERBAIKI dengan rata-rata CER di atas 70%")
    print("\n" + "=" * 100)

# Jalankan aggregasi
aggregate_results()

[OK] Memuat hasil untuk 1.jpg
[OK] Memuat hasil untuk 2.jpg
[OK] Memuat hasil untuk 3.jpg
[OK] Memuat hasil untuk 4.jpg
[OK] Memuat hasil untuk 5.jpg
[OK] Memuat hasil untuk 6.jpg
[OK] Memuat hasil untuk 7.jpg
[OK] Memuat hasil untuk 8.jpg
[OK] Memuat hasil untuk 9.jpg

                          RINGKASAN AKHIR PENGUJIAN OCR MISTRAL (9 GAMBAR)                          

1. JUMLAH GAMBAR YANG DIUJI: 9 dari 9

2. DETAIL HASIL PER GAMBAR:
----------------------------------------------------------------------------------------------------
NAMA FILE       CER          WER         
----------------------------------------------------------------------------------------------------
1.jpg             34.33%     65.31%
2.jpg              2.58%      6.12%
3.jpg             11.59%     40.82%
4.jpg              9.01%     26.53%
5.jpg             26.18%     36.73%
6.jpg              4.29%     14.29%
7.jpg             15.02%     22.45%
8.jpg              8.58%     20.41%
9.jpg              8.15%    